# Зоны ближайшего объекта и индекс ближайшего соседа

В предыдущих разделах мы агрегировали точки по регулярной сетке и по административным единицам. Но часто нужно ответить на другие вопросы:

- к какому объекту ближе всего каждая точка и сколько событий приходится на «зону» каждого объекта?
- распределены ли точки случайно или образуют скопления?

В этом разделе мы:

- построим **диаграмму Вороного** — разбиение территории на зоны ближайшего объекта;
- посчитаем, сколько событий попадает в зону каждого объекта;
- рассчитаем **расстояние до ближайшего соседа** и **индекс ближайшего соседа** (_Nearest Neighbour Index, NNI_), чтобы оценить, кластеризованы ли точки.

Разберём всё на примере станций метро и ДТП в Санкт-Петербурге.

Эти инструменты понадобятся для выполнения третьего проекта.


## 0. Импорт библиотек и подготовка данных

### 0.1 Импорт библиотек


In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree

<p style="color:#26528C; font-style:italic"><strong>scipy.spatial</strong> – модуль библиотеки SciPy для работы с геометрией точек. Класс <code>cKDTree</code> строит пространственный индекс (k-d дерево), который позволяет быстро находить ближайших соседей даже для большого числа точек.</p>


### 0.2 Подготовка данных

Используем файлы из нашего репозитория:

- **spb_admin.gpkg** (слой `okrug`) — границы муниципальных округов Санкт-Петербурга;
- **spb_metro.geojson** — станции метро Санкт-Петербурга (2023);
- **spb_dtp.shp** — ДТП в Санкт-Петербурге за январь 2023 года. Источник: [Карта ДТП](https://dtp-stat.ru)

Сначала прочитаем границы округов и объединим их в один полигон — границу города. Для этого используем `dissolve()`, с которым мы познакомились в разделе [Базовые операции с геометрией](geoprocessing_1.ipynb).


In [ ]:
okrug = gpd.read_file("../data/spb_admin.gpkg", layer="okrug")

city_border = okrug.dissolve()[["geometry"]]

city_border.explore()

Прочитаем данные о станциях метро и ДТП:


In [ ]:
metro = gpd.read_file("../data/spb_metro.geojson")
dtp = gpd.read_file("../data/spb_dtp_shp/spb_dtp.shp")

print("Станций метро:", len(metro))
print("ДТП:", len(dtp))

In [ ]:
m = metro.explore(color="red", tooltip="name")
dtp.explore(m=m, color="black", marker_kwds={"radius": 2})

### 0.3 Перепроецирование данных

Все данные хранятся в географической системе координат (EPSG:4326), где координаты измеряются в градусах. И диаграмма Вороного, и индекс ближайшего соседа основаны на **расстояниях**, а площадь территории нужна нам в квадратных метрах. Поэтому перед анализом перепроецируем данные в подходящую зону UTM.

> На широте Санкт-Петербурга один градус долготы примерно в два раза короче одного градуса широты. Если построить диаграмму Вороного в градусах, границы зон окажутся смещены.


In [ ]:
utm_crs = city_border.estimate_utm_crs()

city_border_utm = city_border.to_crs(utm_crs)
metro_utm = metro.to_crs(utm_crs)
dtp_utm = dtp.to_crs(utm_crs)

utm_crs

## 1. Диаграмма Вороного

**Диаграмма Вороного** — это разбиение пространства на области (полигоны Вороного) вокруг набора точек-генераторов. Каждая область содержит все места, которые ближе к своему генератору, чем к любому другому.

Если генераторы — станции метро, то полигон Вороного показывает **зону ближайшей станции**. Такие зоны часто называют зонами влияния или зонами обслуживания.


### 1.1. Проверка дубликатов

Если две точки-генератора совпадают, для них строится одна общая зона. Проверим, есть ли в данных станции с одинаковыми координатами:


In [ ]:
metro_utm[metro_utm.geometry.duplicated(keep=False)][["name", "address"]]

Некоторые станции представлены несколькими записями с одинаковыми координатами (например, `Дунайская-1` и `Дунайская-2`). Оставим по одной точке на каждое место:


In [ ]:
metro_unique = metro_utm.drop_duplicates(subset="geometry").copy()

print("Было:", len(metro_utm), "→ стало:", len(metro_unique))

### 1.2. Построение диаграммы

Построим диаграмму с помощью метода `voronoi_polygons()`. Параметр `extend_to` расширяет диаграмму так, чтобы она покрывала всю территорию города, а не только область между станциями.


In [ ]:
voronoi = metro_unique.voronoi_polygons(extend_to=city_border_utm.geometry.iloc[0])

voronoi_gdf = gpd.GeoDataFrame(geometry=voronoi, crs=utm_crs)

print("Полигонов:", len(voronoi_gdf))

### 1.3. Обрезка по границе территории

Крайние полигоны выходят далеко за пределы города. Обрежем диаграмму по границе Санкт-Петербурга с помощью `clip()`:


In [ ]:
voronoi_clipped = voronoi_gdf.clip(city_border_utm)

### 1.4. Присоединение атрибутов станций

Полигоны Вороного пока не содержат информации о станциях: метод `voronoi_polygons()` не сохраняет порядок исходных точек. Чтобы узнать, какой станции соответствует каждый полигон, выполним пространственное объединение: каждый полигон **содержит** (`contains`) ровно одну станцию.


In [ ]:
metro_zones = gpd.sjoin(
    voronoi_clipped,
    metro_unique[["name", "geometry"]],
    how="left",
    predicate="contains"
).drop(columns="index_right")

metro_zones.head()

Посмотрим на результат:


In [ ]:
m = metro_zones.explore(
    column="name",
    cmap="tab20",
    legend=False,
    style_kwds={"color": "white", "weight": 1, "fillOpacity": 0.5}
)
metro_unique.explore(m=m, color="black", tooltip="name")

Территория города разбилась на зоны ближайших станций. В центре, где станции расположены плотно, зоны маленькие, на окраинах — огромные.

Важно помнить, что диаграмма Вороного основана на **прямом (евклидовом) расстоянии**. Она не учитывает уличную сеть, реки, мосты и другие барьеры, поэтому показывает ближайшую станцию «по прямой», а не по реальному пути.


## 2. Подсчёт событий в зоне каждого объекта

Определим, сколько ДТП произошло в зоне каждой станции метро. Шаги те же, что и при агрегировании по регулярной сетке в разделе [Агрегирование точечных данных по регулярной сетке](geoprocessing_4.ipynb).


### 2.1. Пространственное объединение

Для каждого ДТП определим, в зону какой станции оно попадает:


In [ ]:
dtp_in_zones = gpd.sjoin(
    dtp_utm,
    metro_zones,
    how="left",
    predicate="within"
)

dtp_in_zones[["address", "name"]].head()

### 2.2. Агрегирование данных

Подсчитаем количество ДТП в зоне каждой станции и присоединим результат к полигонам:


In [ ]:
dtp_count = (
    dtp_in_zones
    .groupby("name")
    .size()
    .reset_index(name="dtp_count")
)

metro_zones = metro_zones.merge(dtp_count, on="name", how="left")

# Зоны, в которых не произошло ни одного ДТП
metro_zones["dtp_count"] = metro_zones["dtp_count"].fillna(0).astype(int)

Какие станции лидируют по числу ДТП?


In [ ]:
metro_zones[["name", "dtp_count"]].sort_values("dtp_count", ascending=False).head()

### 2.3. Учёт площади зон

На первом месте оказались станции на окраинах. Но их зоны занимают десятки и сотни квадратных километров, поэтому в них просто «помещается» больше ДТП. Сравнивать абсолютные значения для зон разной площади некорректно, поэтому рассчитаем **плотность** — число ДТП на 1 км²:


In [ ]:
metro_zones["area_km2"] = metro_zones.geometry.area / 1_000_000
metro_zones["dtp_density"] = metro_zones["dtp_count"] / metro_zones["area_km2"]

metro_zones[["name", "dtp_count", "area_km2", "dtp_density"]].sort_values(
    "dtp_density", ascending=False
).head()

In [ ]:
metro_zones.explore(
    column="dtp_density",
    cmap="OrRd",
    scheme="NaturalBreaks",
    k=5,
    legend=True,
    style_kwds={"color": "white", "weight": 1},
    legend_kwds={"caption": "ДТП на 1 км²"}
)

По числу ДТП лидируют крупные окраинные зоны, а по плотности — небольшие зоны станций в центре (`Сенная`, `пл.Восстания`, `Пушкинская`). Абсолютные значения отвечают на вопрос «у какой станции больше всего событий», а плотность — «где события сконцентрированы сильнее всего». При интерпретации результатов важно понимать, какой из этих вопросов вы задаёте.


## 3. Расстояние до ближайшего соседа

Перейдём от вопроса «где больше событий» к вопросу «как расположены события относительно друг друга». Для этого для каждого ДТП найдём расстояние до **ближайшего другого ДТП**.


### 3.1. Расчёт расстояний

Для поиска ближайших соседей используем `cKDTree` — пространственный индекс из библиотеки SciPy. Он работает с массивом координат, поэтому сначала извлечём координаты точек:


In [ ]:
coords = np.column_stack([dtp_utm.geometry.x, dtp_utm.geometry.y])

coords[:5]

Метод `query()` ищет `k` ближайших точек. Ближайшая точка к каждому ДТП — это оно само (расстояние 0), поэтому запрашиваем двух ближайших соседей (`k=2`) и берём второго:


In [ ]:
tree = cKDTree(coords)
distances, _ = tree.query(coords, k=2)

dtp_utm["nn_dist"] = distances[:, 1]

dtp_utm["nn_dist"].describe()

> Если несколько событий имеют одинаковые координаты (например, несколько случаев по одному адресу), расстояние до ближайшего соседа для них будет равно 0. Это корректно: такие события действительно находятся в одном месте. Не удаляйте нулевые расстояния — иначе вы искусственно завысите среднее расстояние.


In [ ]:
print("ДТП с нулевым расстоянием до соседа:", (dtp_utm["nn_dist"] == 0).sum())

### 3.2. Гистограмма расстояний

Посмотрим на распределение расстояний:


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(dtp_utm["nn_dist"], bins=30, color="#4C72B0", edgecolor="white")

ax.axvline(dtp_utm["nn_dist"].mean(), color="red", linestyle="--",
           label=f"Среднее: {dtp_utm['nn_dist'].mean():.0f} м")
ax.axvline(dtp_utm["nn_dist"].median(), color="black", linestyle=":",
           label=f"Медиана: {dtp_utm['nn_dist'].median():.0f} м")

ax.set_title("Распределение расстояний до ближайшего ДТП")
ax.set_xlabel("Расстояние до ближайшего соседа, м")
ax.set_ylabel("Количество ДТП")
ax.legend()

plt.show()

Распределение сильно скошено вправо: у большинства ДТП ближайшее другое ДТП находится в пределах нескольких сотен метров, и лишь у немногих — в нескольких километрах. Эти «длинные хвосты» — одиночные ДТП на окраинах. Из-за них среднее заметно больше медианы.

Сама по себе гистограмма не говорит, много это или мало. Чтобы ответить на этот вопрос, нужно сравнить наблюдаемые расстояния с теми, которые получились бы при случайном размещении точек.


## 4. Индекс ближайшего соседа (NNI)

### 4.1. Как считается индекс

**Индекс ближайшего соседа** (индекс Кларка–Эванса) сравнивает **наблюдаемое среднее расстояние** до ближайшего соседа с **ожидаемым средним расстоянием** при полностью случайном размещении того же числа точек на той же территории.

- Наблюдаемое среднее расстояние:

$$
\bar{d}_{obs} = \frac{1}{n}\sum_{i=1}^{n} d_i
$$

- Ожидаемое среднее расстояние при случайном размещении:

$$
\bar{d}_{exp} = \frac{1}{2}\sqrt{\frac{A}{n}}
$$

- Индекс:

$$
NNI = \frac{\bar{d}_{obs}}{\bar{d}_{exp}}
$$

где $d_i$ — расстояние от точки $i$ до ближайшей другой точки, $n$ — число точек, $A$ — площадь территории.

Интерпретация:

| Значение | Тип распределения                                                             |
| -------- | ----------------------------------------------------------------------------- |
| NNI < 1  | **кластеризованное** — точки ближе друг к другу, чем при случайном размещении |
| NNI ≈ 1  | **случайное**                                                                 |
| NNI > 1  | **равномерное (регулярное)** — точки «раздвинуты» друг от друга               |


### 4.2. Расчёт индекса


In [ ]:
n = len(dtp_utm)

# Площадь территории в м² (данные в UTM)
area = city_border_utm.geometry.area.iloc[0]

observed_mean = dtp_utm["nn_dist"].mean()
expected_mean = 0.5 * np.sqrt(area / n)

nni = observed_mean / expected_mean

print(f"Число точек: {n}")
print(f"Площадь территории: {area / 1_000_000:.0f} км²")
print(f"Наблюдаемое среднее расстояние: {observed_mean:.0f} м")
print(f"Ожидаемое среднее расстояние: {expected_mean:.0f} м")
print(f"NNI = {nni:.2f}")

NNI меньше 1: в среднем ДТП расположены ближе друг к другу, чем если бы они были разбросаны по городу случайно. Значит, ДТП **кластеризованы** — они концентрируются в центре и вдоль крупных магистралей.


### 4.3. Проверка статистической значимости

Насколько сильно NNI должен отличаться от 1, чтобы мы могли говорить о неслучайности? Для этого рассчитывают **z-оценку**:

$$
z = \frac{\bar{d}_{obs} - \bar{d}_{exp}}{SE}, \qquad SE = \frac{0.26136}{\sqrt{n^2 / A}}
$$

Если $|z| > 1.96$, отличие от случайного распределения статистически значимо (на уровне 5%). Отрицательное значение $z$ указывает на кластеризацию, положительное — на регулярное размещение.


In [ ]:
se = 0.26136 / np.sqrt(n**2 / area)
z = (observed_mean - expected_mean) / se

print(f"z = {z:.2f}")

$z$ намного меньше −1.96, поэтому кластеризацию ДТП нельзя объяснить случайностью.


### 4.4. Влияние границ территории

Ожидаемое расстояние зависит от площади $A$, поэтому **результат зависит от того, как определена территория исследования**. Чем больше площадь при том же числе точек, тем больше ожидаемое расстояние и тем ниже NNI.

Сравним два варианта:

- граница города (её мы использовали выше);
- выпуклая оболочка (_convex hull_) вокруг всех точек.


In [ ]:
hull = dtp_utm.union_all().convex_hull
hull_area = hull.area

expected_mean_hull = 0.5 * np.sqrt(hull_area / n)
nni_hull = observed_mean / expected_mean_hull

print(f"Граница города:     A = {area / 1_000_000:.0f} км², NNI = {nni:.2f}")
print(f"Выпуклая оболочка:  A = {hull_area / 1_000_000:.0f} км², NNI = {nni_hull:.2f}")

In [ ]:
m = gpd.GeoSeries([hull], crs=utm_crs).explore(
    color="orange", style_kwds={"fillOpacity": 0.1}, name="Выпуклая оболочка"
)
city_border_utm.explore(
    m=m, color="blue", style_kwds={"fillOpacity": 0.1}, name="Граница города"
)
dtp_utm.explore(m=m, color="black", marker_kwds={"radius": 2}, name="ДТП")

Выпуклая оболочка оказалась даже больше территории города: она захватывает акваторию Финского залива и земли за пределами города. Граница города, в свою очередь, включает территории, где ДТП практически невозможны (вода, парки, промзоны). Ни один из вариантов не идеален, но вывод о кластеризации не меняется.

Поэтому при расчёте NNI всегда указывайте, как определена территория исследования, и обосновывайте этот выбор.


## Итог

В этом разделе мы познакомились с двумя инструментами анализа точечных данных.

1. **Диаграмма Вороного** разбивает территорию на зоны ближайшего объекта. Мы построили зоны станций метро, обрезали их по границе города и с помощью пространственного объединения подсчитали число ДТП в зоне каждой станции. Зоны имеют разную площадь, поэтому для их сравнения нужно использовать плотность.

2. **Расстояние до ближайшего соседа** и **индекс ближайшего соседа (NNI)** позволяют оценить, кластеризованы ли точки. Мы построили гистограмму расстояний, рассчитали NNI, проверили его значимость и увидели, что результат зависит от выбора территории исследования.

Оба метода основаны на прямом (евклидовом) расстоянии и не учитывают уличную сеть и барьеры. Поэтому данные нужно перепроецировать в метрическую систему координат, а результаты интерпретировать с учётом этого ограничения.
